# Lesson 3: In-Memory Inventory Management Tool
**Munni Khan** · AIM 401 Python Programming for Data Science

Allowed tools only: lists, dictionaries, tuples, if-elif-else, loops, `.get()`, `.append()`, `.items()`. No `def`, classes, imports or file I/O.

## Task 1: Inventory dictionary and order queue

In [1]:
# Warehouse stock levels  ->  key = product name, value = units on the shelf
inventory = {
    "Laptop Charger": 15,
    "Wireless Mouse": 8,
    "HDMI Cable": 25,
    "Mechanical Keyboard": 3,
    "USB Hub": 0,
    "Webcam": 6,
}

In [2]:
# Incoming batch of customer orders  ->  each tuple is (product name, quantity)
order_queue = [
    ("Wireless Mouse", 5),
    ("Mechanical Keyboard", 4),
    ("HDMI Cable", 20),
    ("USB Hub", 2),
    ("Bluetooth Speaker", 1),
    ("Wireless Mouse", 6),
    ("Laptop Charger", 7),
    ("Webcam", 6),
    ("Webcam", 2),
    ("HDMI Cable", 3),
    ("Laptop Charger", 5),
]

In [3]:
# Snapshot of the opening stock so the report can show "before vs after"
opening_stock = inventory.copy()


# Trackers used by Task 3 (updated inside the Task 2 loop)
unfulfilled_items = []      # list of (product, quantity that could not be supplied)
fully_fulfilled_count = 0
partial_count = 0
rejected_count = 0
units_shipped = 0

## Task 2: Process each order (full / partial / out of stock / invalid)

In [4]:
line = "=" * 64
print(line)
print("  KATHMANDU VALLEY DISTRIBUTION HUB - ORDER BATCH PROCESSING")
print(line)
print(f"  Orders in this batch: {len(order_queue)}")
print(line)

order_number = 0
while order_number < len(order_queue):
    product, requested = order_queue[order_number]
    order_number += 1
    tag = f"[Order #{order_number:02d}]"

    # .get() returns None when the product key does not exist at all
    available = inventory.get(product)

    if available is None:
        # Invalid item: the product is not stocked by this warehouse
        print(f"{tag} ALERT   - '{product}' is not a catalogued item. "
              f"{requested} unit(s) rejected.")
        unfulfilled_items.append((product, requested))
        rejected_count += 1

    elif available >= requested:
        # Full fulfillment: enough stock to cover the whole order
        inventory[product] = available - requested
        units_shipped += requested
        fully_fulfilled_count += 1
        print(f"{tag} SUCCESS - Shipped {requested} x {product}. "
              f"Remaining stock: {inventory[product]}")

    elif available > 0:
        # Partial fulfillment: ship what is left, record the shortfall
        shortfall = requested - available
        inventory[product] = 0
        units_shipped += available
        partial_count += 1
        unfulfilled_items.append((product, shortfall))
        print(f"{tag} PARTIAL - Shipped {available} of {requested} x {product}. "
              f"Back-ordered: {shortfall}")

    else:
        # Out of stock: the key exists but the shelf is empty
        print(f"{tag} ALERT   - '{product}' is OUT OF STOCK. "
              f"{requested} unit(s) could not be supplied.")
        unfulfilled_items.append((product, requested))
        rejected_count += 1

  KATHMANDU VALLEY DISTRIBUTION HUB - ORDER BATCH PROCESSING
  Orders in this batch: 11
[Order #01] SUCCESS - Shipped 5 x Wireless Mouse. Remaining stock: 3
[Order #02] PARTIAL - Shipped 3 of 4 x Mechanical Keyboard. Back-ordered: 1
[Order #03] SUCCESS - Shipped 20 x HDMI Cable. Remaining stock: 5
[Order #04] ALERT   - 'USB Hub' is OUT OF STOCK. 2 unit(s) could not be supplied.
[Order #05] ALERT   - 'Bluetooth Speaker' is not a catalogued item. 1 unit(s) rejected.
[Order #06] PARTIAL - Shipped 3 of 6 x Wireless Mouse. Back-ordered: 3
[Order #07] SUCCESS - Shipped 7 x Laptop Charger. Remaining stock: 8
[Order #08] SUCCESS - Shipped 6 x Webcam. Remaining stock: 0
[Order #09] ALERT   - 'Webcam' is OUT OF STOCK. 2 unit(s) could not be supplied.
[Order #10] SUCCESS - Shipped 3 x HDMI Cable. Remaining stock: 2
[Order #11] SUCCESS - Shipped 5 x Laptop Charger. Remaining stock: 3


## Task 3: End-of-batch summary report

In [5]:
print()
print(line)
print("  END-OF-BATCH SUMMARY REPORT")
print(line)

# 1. Final updated inventory dictionary
print("1) Final updated inventory dictionary:")
print(f"   {inventory}")
print()
print(f"   {'Product':<22}{'Opening':>9}{'Closing':>9}{'Moved':>8}")
print(f"   {'-' * 48}")
for product, closing in inventory.items():
    opening = opening_stock[product]
    status = "  <- restock" if closing == 0 else ""
    print(f"   {product:<22}{opening:>9}{closing:>9}{opening - closing:>8}{status}")

# 2. Total count of fully fulfilled orders
print()
print(f"2) Fully fulfilled orders: {fully_fulfilled_count} of {len(order_queue)}")
print(f"   Partially fulfilled    : {partial_count}")
print(f"   Not supplied at all    : {rejected_count}")
print(f"   Total units shipped    : {units_shipped}")

# 3. Items / quantities that could not be fully supplied
print()
print("3) Items that could not be fully supplied:")
if len(unfulfilled_items) == 0:
    print("   None - every order was completed in full.")
else:
    total_short = 0
    for product, quantity in unfulfilled_items:
        print(f"   - {product:<22} short by {quantity} unit(s)")
        total_short += quantity
    print(f"   Total units not supplied: {total_short}")
    print(f"   Raw list: {unfulfilled_items}")
print(line)


  END-OF-BATCH SUMMARY REPORT
1) Final updated inventory dictionary:
   {'Laptop Charger': 3, 'Wireless Mouse': 0, 'HDMI Cable': 2, 'Mechanical Keyboard': 0, 'USB Hub': 0, 'Webcam': 0}

   Product                 Opening  Closing   Moved
   ------------------------------------------------
   Laptop Charger               15        3      12
   Wireless Mouse                8        0       8  <- restock
   HDMI Cable                   25        2      23
   Mechanical Keyboard           3        0       3  <- restock
   USB Hub                       0        0       0  <- restock
   Webcam                        6        0       6  <- restock

2) Fully fulfilled orders: 6 of 11
   Partially fulfilled    : 2
   Not supplied at all    : 3
   Total units shipped    : 52

3) Items that could not be fully supplied:
   - Mechanical Keyboard    short by 1 unit(s)
   - USB Hub                short by 2 unit(s)
   - Bluetooth Speaker      short by 1 unit(s)
   - Wireless Mouse         short by 